### refine text to answering rag

In [2]:
from typing import List, TypedDict
import time
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from typing import Union
from pydantic import Field

C:\Users\arghy\AppData\Local\Temp\ipykernel_6968\1159099786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [3]:
result = retriever.invoke('what is the syllabus of information technology humanities')

In [4]:
for i in range(len(result)):
    print(result[i].page_content)

Maulana Abul Kalam Azad University of Technology, West Bengal   
(Formerly West Bengal University of Technology)  
Syllabus for B. Tech in Information Technology   
(Applicable from the academic session 2018-2019)   
10   
PG   
   
 
Text books/ reference books:   
   
1. John E. Hopcroft, Rajeev Motwani and Jeffrey D. Ullman, Introduction to Automata Theory, 
Languages, and Computation, Pearson Education Asia.   
2. Harry R. Lewis and Christos H. Papadimitriou, Elements of the Theory of Computation, 
Pearson Education Asia.   
3. Dexter C. Kozen, Automata and Computability, Undergraduate Texts in Computer Science, 
Springer.   
4. Michael Sipser, Introduction to the Theory of Computation, PWS Publishing.   
5. John Martin, Introduction to Languages and The Theory of Computation, TataMcGraw Hill., 
PEARSON.   
6. Dr. R.B. Patel, Theory of Computation, Khanna Publishing House
Maulana Abul Kalam Azad University of Technology, West Bengal   
(Formerly West Bengal University of Technology

In [4]:
print(type(result[i].page_content))

<class 'str'>


In [5]:
import re
def decompose_to_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]

In [6]:
strip_list = []
count = 0
for i in range(len(result)):
    z=(decompose_to_sentences(result[i].page_content))
    for j in range(len(z)):
        strip_list.append(z[j])

In [7]:
query = 'syllabus of 3rd semister'
class KeepOrDrop(BaseModel):
    keepordrop:bool = Field(description='return True if the strip is important to answer as per query and False if the sentence is not that important for the query')
llm = ChatOllama(model='qwen2.5:7b',temperature=0)
llm_keepordrop = llm.with_structured_output(KeepOrDrop)
dropping_list = []
for i in range(len(strip_list)):
    z = llm_keepordrop.invoke(f'query:{query},sentence:{strip_list[i]}')
    if not z.keepordrop:
        dropping_list.append(i)

In [8]:
refined_text = ''
for i in range(len(strip_list)):
    if i not in dropping_list:
        refined_text = refined_text + '\n' + strip_list[i]

In [9]:
print(refined_text)


Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
Tech in Information Technology (Applicable from the academic session 2018-2019) 10 PG Text books/ reference books: 1.
Hopcroft, Rajeev Motwani and Jeffrey D.
Ullman, Introduction to Automata Theory, Languages, and Computation, Pearson Education Asia.
Papadimitriou, Elements of the Theory of Computation, Pearson Education Asia.
Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
2 Design an Adder/Subtractor composite unit.
Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
Tech in Information Technology (Applicable from the academic session 2018-2019) 19 PG Design & Analysis Algorithm Lab Code: PC-IT492 Contact: 4P Name of the Course: Design & Analysis Algorithm Lab Course Code: PC-IT492 Semester: IV Duration:6 mont

In [10]:
result = llm.invoke(f'query:{query} and retrieved text:{refined_text}')

In [11]:
print(result.content)

Based on the retrieved text, it appears that the syllabus for the third semester of the B.Tech in Information Technology program at Maulana Abul Kalam Azad University of Technology, West Bengal (formerly West Bengal University of Technology) is not fully provided. However, the text does include details about a few courses and labs. Here's a summary of the information available:

### Course Details:
1. **Design & Analysis of Algorithms Lab (PC-IT492)**
   - **Semester:** IV
   - **Duration:** 6 months
   - **Maximum Marks:** 100
   - **Teaching Scheme:**
     - Theory: 4 hours/week
     - Tutorial: NIL
     - Practical: 4 hours/week
   - **Distribution of Marks:**
     - Continuous Internal Assessment: 40 marks
     - External Assessment: 60 marks
   - **Credit Points:** 2
   - **Course Outcomes:** PC-IT404.1, PC-IT404.2, PC-IT404.3
   - **Pre-Requisite:** Not specified in the provided text

2. **Environmental Sciences (MC-IT401)**
   - **Semester:** IV
   - **Duration:** 6 months
   - 

### retrieving till llm does not think that ans is good 

In [ ]:
from typing import List, TypedDict,Annotated
import time
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langgraph.graph.message import add_messages, BaseMessage
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from typing import Union,Literal
from pydantic import Field
import re
retrieve_again = '''You are a strict retrieval quality evaluator.

Your job is to determine whether the retrieved documents are sufficient
and correctly matched to the user's query.

Return "yes" if another retrieval is necessary.

Return "yes" when:
- The retrieved documents do not contain enough information.
- The retrieved documents contain information from the wrong semester.
- The retrieved documents contain information from the wrong branch.
- Important subjects or requested fields are missing.
- The retrieved documents contain mixed or conflicting information.
- A more targeted search could improve accuracy.

Return "no" ONLY when:
- The retrieved information directly matches the requested semester/branch/topic.
- The important requested information is present.
- There is no obvious irrelevant or conflicting semester information.
- The retrieved documents are sufficient to answer accurately.

For queries asking for ALL subjects, verify that the retrieved
information appears complete rather than assuming that a few subjects
are sufficient.

Return only "yes" or "no".
'''
sentence_filter_prompt = '''You are a lenient relevance filter.

You will be given a query and a numbered list of sentences.

For each sentence, return keep=True if the sentence has ANY relevant or overlapping context with the query — 
even partial, indirect, or loosely related information counts as True.

Return keep=False ONLY if the sentence is about a totally different, unrelated topic 
with no connection to the query at all.

When in doubt, return True. Only drop sentences that are clearly and completely irrelevant.

Return a decision for every sentence, using its index.'''
class SentenceDecision(BaseModel):
    index: int = Field(description='the index of the sentence')
    keep: bool = Field(description='True if sentence has any relevant/overlapping context with the query, False only if totally unrelated')

class KeepOrDropBatch(BaseModel):
    decisions: List[SentenceDecision]
llm = ChatOllama(model='qwen2.5:7b',temperature=0)
# llm_keepordrop = llm.with_structured_output(KeepOrDrop)
llm_keepordrop_batch = llm.with_structured_output(KeepOrDropBatch)

class enough_or_not(BaseModel):
    enoughornot: Literal['yes','no'] = Field(description=f'{retrieve_again}')
llm_retrieve_again = llm.with_structured_output(enough_or_not)



path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})
class RagState(TypedDict):
    query:Annotated[list, add_messages]
    refined_retrieved_text:str
    answer:str
    retrieve_again:str
    retry_count:int

def retrieve_to_refine(state:RagState):
    query = state['query'][-1].content
    strip_list = []
    previous_text = state['refined_retrieved_text']
    result = retriever.invoke(f'{query}')
    for i in range(len(result)):
        z = decompose_to_sentences(result[i].page_content)
        for j in range(len(z)):
            strip_list.append(z[j])

    numbered = '\n'.join(f'{i}: {s}' for i, s in enumerate(strip_list))
    batch_result = llm_keepordrop_batch.invoke(
        f'{sentence_filter_prompt}\n\nquery: {query}\n\nsentences:\n{numbered}'
    )

    keep_indices = {d.index for d in batch_result.decisions if d.keep}
    refined_text = '\n'.join(strip_list[i] for i in keep_indices if i < len(strip_list))

    total = previous_text + '\n' + refined_text
    return {'refined_retrieved_text': total, 'retry_count': state.get('retry_count',0) + 1}
def enough(state:RagState):
    query =  state['query'][-1].content
    text = state['refined_retrieved_text']
    yes_or_no = llm_retrieve_again.invoke(f'query:{query} and retrieved docs are:\n {text}')
    return {'retrieve_again':yes_or_no.enoughornot}

def route(state:RagState):
    if state['retrieve_again'].lower() == 'yes':
        return 'new_query'
    elif state['retrieve_again'].lower() == 'no':
        return 'generate'

def new_query(state:RagState):
    new_query = llm.invoke(f"""
You are a query rewriting agent for a PDF retrieval system.

The human has provided the following query:

{state["query"]}

Rewrite this query into a new, more precise search query that can retrieve additional relevant information from the PDF.

Requirements:
- Preserve the original intent of the human's query.
- Identify the key concepts, entities, keywords, and context.
- Add useful related terms that may appear in the PDF.
- Make the query more specific and retrieval-friendly.
- Do not change the meaning of the original query.
- Do not answer the question.
- Return only the rewritten query, with no explanation.
""")
    return {'query':HumanMessage(content=new_query.content)}
def generate(state:RagState):
    result = llm.invoke(f'generate answer as per the query:{state["query"][0]} and the retrieve documents:{state["refined_retrieved_text"]}').content
    return {'answer':result}
def decompose_to_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text)]
graph = StateGraph(RagState)
graph.add_node('retrieve',retrieve_to_refine)
graph.add_node('enough',enough)
graph.add_node('generate',generate)
graph.add_node('new_query',new_query)

graph.add_edge(START,'retrieve')
graph.add_edge('retrieve','enough')
graph.add_conditional_edges('enough',route)
graph.add_edge('new_query','retrieve')
graph.add_edge('generate',END)
workflow = graph.compile()
while True:
    query = input('USER:')
    if query.lower() in ['exit','bye','stop']:
        break
    result = workflow.invoke({'query':query,'refined_retrieved_text':''})
    print('AI:',result['answer'])

C:\Users\arghy\AppData\Local\Temp\ipykernel_20376\231592850.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


AI: {'query': [HumanMessage(content='what is the name of the college', additional_kwargs={}, response_metadata={}, id='d61a86ab-7845-4f04-90ad-419ea1f89674'), HumanMessage(content='what is the name of the college where the research was conducted', additional_kwargs={}, response_metadata={}, id='ba61628e-24e2-4efa-a9a7-041fc497b4e0'), HumanMessage(content='what is the name of the institution where the research was conducted', additional_kwargs={}, response_metadata={}, id='643bb81e-785a-4cdb-975c-1f65a5074015'), HumanMessage(content='what is the name of the institution or college where the research was conducted including keywords like university, academy, or institute', additional_kwargs={}, response_metadata={}, id='f420d194-af12-4658-be9b-ca0cb6a76d29'), HumanMessage(content='what is the name of the university, academy, or institute where the research was conducted', additional_kwargs={}, response_metadata={}, id='ef74443d-b2ec-4c96-acd6-5ed6f70d83a8'), HumanMessage(content='what is 

In [2]:
result['answer']

'The name of the college mentioned in the provided content is Jalpaiguri Government Engineering College.'

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)
result = llm.invoke('what is the workof you')
print(result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


content=[{'type': 'text', 'text': 'I am an AI (Artificial Intelligence) assistant. My "work" is to process information, answer questions, and assist you with a wide variety of tasks. \n\nHere are some of the main things I can help you with:\n\n* **Answering Questions:** Explaining complex topics, giving quick facts, or helping with general knowledge on almost any subject.\n* **Writing & Editing:** Drafting emails, essays, stories, reports, resumes, or blog posts, as well as checking grammar and improving text.\n* **Brainstorming:** Generating ideas for projects, gifts, business names, marketing strategies, or creative endeavors.\n* **Coding & Tech Support:** Writing code, finding bugs in your programs, and explaining technical concepts in languages like Python, JavaScript, C++, etc.\n* **Summarizing & Translating:** Turning long articles or documents into short summaries, and translating text between different languages.\n* **Learning & Tutoring:** Helping you practice a new language, 

In [6]:
result.content[0]['text']

'I am an AI (Artificial Intelligence) assistant. My "work" is to process information, answer questions, and assist you with a wide variety of tasks. \n\nHere are some of the main things I can help you with:\n\n* **Answering Questions:** Explaining complex topics, giving quick facts, or helping with general knowledge on almost any subject.\n* **Writing & Editing:** Drafting emails, essays, stories, reports, resumes, or blog posts, as well as checking grammar and improving text.\n* **Brainstorming:** Generating ideas for projects, gifts, business names, marketing strategies, or creative endeavors.\n* **Coding & Tech Support:** Writing code, finding bugs in your programs, and explaining technical concepts in languages like Python, JavaScript, C++, etc.\n* **Summarizing & Translating:** Turning long articles or documents into short summaries, and translating text between different languages.\n* **Learning & Tutoring:** Helping you practice a new language, study for an exam, or understand a

In [3]:
path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':10})

In [5]:
query = 'what is the syllabus of all subject in 3rd semister'
retrieved_data = retriever.invoke(query)
llm = ChatOllama(model = 'qwen2.5:7b')
ans = llm.invoke(f'query:{query}\nretrieved data:\n{retrieved_data}')
print(ans.content)

Based on the retrieved data, the syllabus for the 3rd semester of B. Tech in Information Technology at Maulana Abul Kalam Azad University of Technology, West Bengal (formerly West Bengal University of Technology) includes a variety of subjects and practical components. Here is a summary:

### Theory Subjects:
1. **Programming in R**
   - Introduction to R programming, installation and running R, R help files, R sessions, R objects (vectors, attributes, matrices, arrays, classes, lists, data frames), and operators in R.

2. **Communication**
   - 3-0-0-3 (3 credits)
   - Project-III (0-0-12-12, 6 credits)
   - Open Elective-III (Operations Research, Mobile Computing, Robotics, Microwave)
   - Project-II (0-0-12-12, 6 credits)
   - Viva (0-0-0-0, 2 credits)
   - Internship Evaluation (0-0-0-0, 0 credits)
   - Total Credits: 21

3. **Analog & Digital Electronics (ES-IT301)**
   - Course Code: ES-IT301
   - Duration: 6 months
   - Maximum Marks: 100
   - Teaching Scheme: Theory: 3 hrs./wee